In [1]:
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta

from sentinelhub import (
    SHConfig,
    SentinelHubRequest,
    DataCollection,
    BBox,
    CRS,
    MimeType
)

config = SHConfig()
config.sh_client_id = ""
config.sh_client_secret = ""
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.download_timeout_seconds = 600.0

print("Autenticación configurada.")

Autenticación configurada.


In [2]:
amg_bbox = BBox(bbox=[-90.76260926, 14.24228736, -90.21187736, 14.91470121], crs=CRS.WGS84)
print("Bounding Box de Guatemala inicializado.")

s5p_ch4_collection = DataCollection.SENTINEL5P.define_from(
    "sentinel-5p-l2", 
    service_url=config.sh_base_url
)
print("Colección Sentinel-5P inicializada correctamente.")

Bounding Box de Guatemala inicializado.
Colección Sentinel-5P inicializada correctamente.


In [3]:
evalscript_ch4 = """//VERSION=3
function setup() {
    return {
        input: ["CH4", "dataMask"],
        output: { bands: 2, sampleType: "FLOAT32" },
        mosaicking: "ORBIT"
    };
}

function isValid(sample) {
    return sample.dataMask === 1;
}

function sum(array) {
    let total = 0;
    for (let i = 0; i < array.length; i++) {
        total += array[i].CH4;
    }
    return total;
}

function evaluatePixel(samples) {
    const validSamples = samples.filter(isValid);
    if (validSamples.length === 0) {
        return [0, 0]; 
    }
    const mean = sum(validSamples) / validSamples.length;
    return [mean, 1];
}
"""

In [4]:
def download_monthly(start_date, end_date, folder_name="CH4_Historico"):
    request = SentinelHubRequest(
        evalscript=evalscript_ch4,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=s5p_ch4_collection,
                time_interval=(start_date, end_date),
                other_args={"processing": {"minQa": 50}}
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=amg_bbox,
        resolution=(0.0629, 0.0494),
        config=config,
        data_folder=folder_name #
    )
    
    request.get_data(save_data=True)

os.makedirs("CH4_Historico", exist_ok=True)

start_period = datetime(2018, 7, 1)
end_period = datetime(2025, 7, 1)
current_date = start_period

print("Iniciando extracción masiva de CH4...")

while current_date < end_period:
    next_month = current_date + relativedelta(months=1)
    end_date = next_month - relativedelta(days=1)
    
    str_start = current_date.strftime("%Y-%m-%d")
    str_end = end_date.strftime("%Y-%m-%d")
    
    print(f" -> Procesando mes: {str_start} al {str_end}")
    
    try:
        download_monthly(str_start, str_end)
    except Exception as e:
        print(f"    [!] Error al procesar {str_start}: {e}")
        
    current_date = next_month

print(" extracción completado exitosamente!")

Iniciando extracción masiva de CH4...
 -> Procesando mes: 2018-07-01 al 2018-07-31
 -> Procesando mes: 2018-08-01 al 2018-08-31
 -> Procesando mes: 2018-09-01 al 2018-09-30
 -> Procesando mes: 2018-10-01 al 2018-10-31
 -> Procesando mes: 2018-11-01 al 2018-11-30
 -> Procesando mes: 2018-12-01 al 2018-12-31
 -> Procesando mes: 2019-01-01 al 2019-01-31
 -> Procesando mes: 2019-02-01 al 2019-02-28
 -> Procesando mes: 2019-03-01 al 2019-03-31
 -> Procesando mes: 2019-04-01 al 2019-04-30
 -> Procesando mes: 2019-05-01 al 2019-05-31
 -> Procesando mes: 2019-06-01 al 2019-06-30
 -> Procesando mes: 2019-07-01 al 2019-07-31
 -> Procesando mes: 2019-08-01 al 2019-08-31
 -> Procesando mes: 2019-09-01 al 2019-09-30
 -> Procesando mes: 2019-10-01 al 2019-10-31
 -> Procesando mes: 2019-11-01 al 2019-11-30
 -> Procesando mes: 2019-12-01 al 2019-12-31
 -> Procesando mes: 2020-01-01 al 2020-01-31
 -> Procesando mes: 2020-02-01 al 2020-02-29
 -> Procesando mes: 2020-03-01 al 2020-03-31
 -> Procesando me

In [5]:
import os
import json
import shutil

carpeta_cache = "CH4_Historico"
carpeta_limpia = "CH4_Mensual"
os.makedirs(carpeta_limpia, exist_ok=True)

print("Iniciando organización de archivos...")

for folder_hash in os.listdir(carpeta_cache):
    folder_path = os.path.join(carpeta_cache, folder_hash)
    
    if os.path.isdir(folder_path):
        json_path = os.path.join(folder_path, "request.json")
        tiff_path = os.path.join(folder_path, "response.tiff")
        
        if os.path.exists(json_path) and os.path.exists(tiff_path):
            with open(json_path, 'r') as f:
                data = json.load(f)
                
            try:
                payload = data.get("request", {}).get("payload", {})
                time_range = payload.get("input", {}).get("data", [{}])[0].get("dataFilter", {}).get("timeRange", {})
                fecha_inicio = time_range.get("from", "")
                
                if fecha_inicio:
                    mes_año = fecha_inicio.split("-")[0] + "_" + fecha_inicio.split("-")[1]
                    nuevo_nombre = f"CH4_{mes_año}.tiff"
                    
                    nuevo_path = os.path.join(carpeta_limpia, nuevo_nombre)
                    shutil.copy(tiff_path, nuevo_path)
                    print(f" -> {folder_hash[:8]}... organizado como {nuevo_nombre}")
                
            except Exception as e:
                print(f"Error procesando {folder_hash}: {e}")

print(f"¡Listo! Revisa la carpeta '{carpeta_limpia}'.")

Iniciando organización de archivos...
 -> e629ce6b... organizado como CH4_2018_07.tiff
 -> 65b087f3... organizado como CH4_2018_08.tiff
 -> ac8c15da... organizado como CH4_2018_09.tiff
 -> f6c30c18... organizado como CH4_2018_10.tiff
 -> 97cbf62d... organizado como CH4_2018_11.tiff
 -> 751e94ca... organizado como CH4_2018_12.tiff
 -> d5d89cc2... organizado como CH4_2019_01.tiff
 -> 7769bc02... organizado como CH4_2019_02.tiff
 -> 9558f30a... organizado como CH4_2019_03.tiff
 -> 5b335861... organizado como CH4_2019_04.tiff
 -> aab2e402... organizado como CH4_2019_05.tiff
 -> dc49aab3... organizado como CH4_2019_06.tiff
 -> d71b090f... organizado como CH4_2019_07.tiff
 -> 78a5712c... organizado como CH4_2019_08.tiff
 -> 0a2a8185... organizado como CH4_2019_09.tiff
 -> 27ba3529... organizado como CH4_2019_10.tiff
 -> ea84798c... organizado como CH4_2019_11.tiff
 -> ed1f6bf6... organizado como CH4_2019_12.tiff
 -> cd25832e... organizado como CH4_2020_01.tiff
 -> 7220fd72... organizado como

In [6]:
carpeta_origen = "mystorage/CH4_Mensual"
nombre_archivo_destino = "CH4_Historico_Respaldo"

print("Iniciando compresión, esto puede tomar unos segundos...")

try:
    shutil.make_archive(nombre_archivo_destino, 'zip', carpeta_origen)
    print(f"¡Éxito! El archivo {nombre_archivo_destino}.zip está listo para descargar.")
except FileNotFoundError:
    print(f"Error: No se encontró la carpeta {carpeta_origen}. Revisa la ruta.")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

Iniciando compresión, esto puede tomar unos segundos...
¡Éxito! El archivo CH4_Historico_Respaldo.zip está listo para descargar.
